# 分子系統解析入門１　〜鳥類の系統樹を作ろう〜

## 概要
**31種類の脊椎動物**（30種の鳥類 + 外群としてミシシッピワニ）のミトコンドリアDNA（cytochrome b）から系統樹 (phylogenetic tree)を推定します。
このNotebookでは系統解析を以下の2段構えで行います。
1. **入門編** — ざっくりとしたアプローチ（切り詰めアラインメント + 近隣結合法）で大まかな系統解析の流れを掴みます
1. **本格編** — 実際の研究の現場で標準的に使われているツール、MAFFTでアラインメントし、IQ-TREEで最尤系統樹を推定します

### 学習目標
1. 遺伝子配列データ（FASTA形式）を操作する
2. 進化距離という考え方を理解する
3. 近隣結合法（NJ法）と最尤法（ML法）をそれぞれ動かしてみる
4. ブートストラップで内部枝の信頼度を測る
5. 作った樹を読み解き、鳥類の進化系統について考察する

### 系統樹とは：進化の歴史を「木」の形で表したもの。
- 枝の長さが進化的な距離を反映
- 分岐パターンが共通祖先からの分かれ方を示す
- 末端(葉)が現存する種、内部の分岐点が「共通祖先」を表す

### なぜミトコンドリアDNAのcytb遺伝子を解析するのか?

ミトコンドリア DNA は母系遺伝で、種を超えて比較しやすい性質を持っています。
中でもcytochrome b (cytb)は脊椎動物で約 1140 塩基 / 380 アミノ酸とほぼ長さが保存され、
種の系統解析に古くから使われてきた「定番の遺伝子」です。

### Licence
- 配列データ：NCBI のパブリックデータ
- シルエット画像：PhyloPic(CC ライセンス、`birds/phylopic/phylopic_attribution.tsv` 参照)。

## 0. 必要ライブラリのインストール(初回のみ)

Python パッケージをインストールします。

- `biopython` : 配列解析と系統樹計算
- `matplotlib-fontja` : 日本語フォント
- `ete3` + `PyQt5` : 系統樹の描画(本ノートブックの樹はすべて ETE3 で描きます)

注意点
- `MAFFT`と`IQ-TREE`は別途インストールが必要です（後のセクション 12 で扱います）。
- ETE3 は内部で Qt を使って樹を画像に描き出します。画面のないサーバーや Colab で動かす場合は、次のセルで環境変数 `QT_QPA_PLATFORM=offscreen` を設定して **ヘッドレス描画** に切り替えます。

In [ ]:
%pip install biopython pandas numpy matplotlib seaborn matplotlib-fontja ete3 PyQt5

## 1. ライブラリの読み込みと日本語フォントの設定

グラフに日本語を出すために `matplotlib-fontja` を使います（seaborn のスタイル設定を先に行い、その後に `matplotlib_fontja` を読み込みます）。

In [ ]:
import os
import sys
import shutil
import tempfile
import subprocess
import warnings
from pathlib import Path

# ETE3 はヘッドレス環境(画面なしのサーバー・Colab)でも描画できるよう、
# Qt のインポートより前に offscreen プラットフォームを指定しておく
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
import seaborn as sns

from Bio import SeqIO, AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

# 系統樹の描画は ETE3 で行う
from ete3 import Tree as EteTree, TreeStyle, TextFace, ImgFace, NodeStyle, RectFace
from IPython.display import Image as IPyImage, display

# 日本語フォントの設定(順序が重要)
sns.set_style("whitegrid")
try:
    import matplotlib_fontja  # noqa: F401
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError:
    print("matplotlib-fontja が見つかりません。上のセルでインストールしてください。")
plt.rcParams["axes.unicode_minus"] = False

# ETE3 の TextFace で日本語を表示するためのフォント名(CJK を含むものを指定)
JP_FONT = "Noto Sans CJK JP"

## 2. FASTA ファイルの読み込み

**FASTA** は配列データを表す最も基本的なテキスト形式です。

```
>Struthio_camelus|NC_002785.1|CYTB    ← ヘッダー(`>` で始まる)
ATGGCCCCCAACATTCGAAAATCG...           ← 配列本体
```

このリポジトリのヘッダーは `>属_種|NCBI アクセッション番号|遺伝子名` の3項目を `|` で区切った形式です。

In [ ]:
# データファイルへのパス(このノートブックは ml/ から実行する想定)
BIRDS_DIR = Path("birds")
FASTA_PATH = BIRDS_DIR / "birds_cytb.fasta"
METADATA_PATH = BIRDS_DIR / "birds_cytb_metadata.tsv"
PHYLOPIC_DIR = BIRDS_DIR / "phylopic" / "png"

# MAFFT / IQ-TREE 用の作業ディレクトリ
WORK_DIR = Path("./phylo_work")
WORK_DIR.mkdir(exist_ok=True)

for p in [FASTA_PATH, METADATA_PATH, PHYLOPIC_DIR]:
    assert p.exists(), f"見つかりません: {p}"
print(f"データ準備 OK: {BIRDS_DIR.resolve()}")
print(f"作業ディレクトリ: {WORK_DIR.resolve()}")

records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print(f"読み込んだ配列数: {len(records)}")
print(f"最初のヘッダー : {records[0].id}")
print(f"最初の60塩基   : {str(records[0].seq)[:60]} ...")

# ヘッダーから「種名(属_種)」だけを取り出して ID を整理
for rec in records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""

print(f"\n整理後の ID 一覧(先頭5件): {[r.id for r in records[:5]]}")

## 3. メタデータ（和名・分類群）の読み込み

各種の和名・英名・分類群(目・系統)が `birds_cytb_metadata.tsv` にまとめられています。

- **目 (order)** : 古典的な分類単位(例: ペリカン目 Pelecaniformes)
- **系統 (clade)** : 大きな進化グループ
    - **Palaeognathae** : 古顎類 — ダチョウ、キーウィなど飛べない走鳥
    - **Galloanserae** : キジカモ類 — ニワトリ、カモなど
    - **Neoaves** : 新鳥類 — 上記以外の鳥類すべて(現生鳥類の約95%)
    - **Outgroup** : 外群(今回はミシシッピワニ)

**外群 (outgroup)** とは、解析対象から「最も離れている」と分かっている種で、系統樹の「根」を決めるのに使います。
鳥類はワニと共通祖先を持つ(両者は爬虫類のグループの一部)ことが化石記録などから分かっているので、ワニは外群として適切です。

In [ ]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
# FASTA の ID(アンダースコア形式)と紐付けるキー
metadata["id"] = metadata["scientific_name"].str.replace(" ", "_")

print(f"メタデータ行数: {len(metadata)}")
print("\n各系統の種数:")
print(metadata["clade"].value_counts())
metadata.head()

## 4. 配列の長さと組成を確認

解析に入る前に、データの状態をチェックします。

- **配列長** がそろっているか? → そろっていなければ後でアラインメントが必要
- **GC 含量** (G+Cの割合) → 種ごとに傾向があり、ミトコンドリアでは低めの傾向

In [ ]:
def gc_content(seq):
    s = str(seq).upper()
    return (s.count("G") + s.count("C")) / len(s) * 100

seq_stats = pd.DataFrame({
    "id": [r.id for r in records],
    "length": [len(r.seq) for r in records],
    "gc_pct": [gc_content(r.seq) for r in records],
}).merge(metadata[["id", "common_name_ja", "clade"]], on="id", how="left")

print("配列長の統計:")
print(seq_stats["length"].describe().round(1))
print(f"\n配列長の種類: {sorted(seq_stats['length'].unique())}")
seq_stats.head()

In [ ]:
# 系統別の GC 含量(箱ひげ図 + 個々の点)
fig, ax = plt.subplots(figsize=(9, 4))
order = seq_stats.groupby("clade")["gc_pct"].median().sort_values().index
sns.boxplot(data=seq_stats, x="clade", y="gc_pct", order=order,
            color="lightsteelblue", ax=ax)
sns.stripplot(data=seq_stats, x="clade", y="gc_pct", order=order,
              color="darkblue", alpha=0.6, size=4, ax=ax)
ax.set_xlabel("系統 (clade)")
ax.set_ylabel("GC 含量 (%)")
ax.set_title("cytb 遺伝子の GC 含量(系統別)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. 入門編 — シンプルなアラインメント

DNA から系統樹を作るには、**「同じ位置に同じ進化的起源の塩基を縦に揃える」** 作業が必要です。
これを **マルチプルアラインメント (multiple sequence alignment, MSA)** と呼びます。

今回のデータには都合のいい特徴があります。

- すべての配列が **ATG**(開始コドン)から始まっている
- 長さは 1140 〜 1159 塩基とほぼ同じ(脊椎動物の cytb は長さがよく保存されている)
- 違いは主に末尾(終止コドン周辺の数塩基)

そこで入門編では、簡単のため **全配列の先頭から共通の長さだけを切り出して使う** ことにします。
本格的なアラインメントは後のセクション 12 で MAFFT を使って行います。

### 5.1 切り詰めアライメント

In [ ]:
# 全配列の最小長まで切り詰める(=先頭から揃える)
min_len = min(len(r.seq) for r in records)
print(f"全配列を {min_len} 塩基に切り詰めます")

for rec in records:
    rec.seq = rec.seq[:min_len]

alignment = MultipleSeqAlignment(records)
print(f"\nアラインメント:")
print(f"  配列数 : {len(alignment)}")
print(f"  カラム数: {alignment.get_alignment_length()}")

### 5.2 進化距離の計算

種同士が進化的にどれだけ「離れている」かを数値化します。
今回は最もシンプルな **p距離** を使います。

$$
d_{\mathrm{p}}(A, B) = \frac{\text{異なる塩基の数}}{\text{全塩基数}}
$$

たとえば 1140 塩基中 100 ヶ所違えば $d = 100/1140 \approx 0.088$。

In [ ]:
calculator = DistanceCalculator("identity")
dm = calculator.get_distance(alignment)

print(f"距離行列のサイズ: {len(dm.names)} × {len(dm.names)}")
print(f"\n例:")
print(f"  ダチョウ vs キーウィ(どちらも古顎類)")
print(f"    d = {dm['Struthio_camelus', 'Apteryx_rowi']:.4f}")
print(f"  ダチョウ vs ニワトリ(別系統)")
print(f"    d = {dm['Struthio_camelus', 'Gallus_gallus']:.4f}")
print(f"  ダチョウ vs ミシシッピワニ(外群)")
print(f"    d = {dm['Struthio_camelus', 'Alligator_mississippiensis']:.4f}")

### 5.3 距離行列をヒートマップで可視化

行と列を **系統(clade)順** に並べて、進化距離を俯瞰してみます。ざっくりと３つのグループに別れることが観察できるでしょうか？

In [ ]:
names = list(dm.names)
n = len(names)
mat = np.array([[dm[a, b] for b in names] for a in names])
dist_df = pd.DataFrame(mat, index=names, columns=names)

# clade 順に並べ替え(Palaeognathae → Galloanserae → Neoaves → Outgroup)
clade_order_list = ["Palaeognathae", "Galloanserae", "Neoaves", "Outgroup"]
id_to_clade = metadata.set_index("id")["clade"].to_dict()
ordered = sorted(names, key=lambda x: (clade_order_list.index(id_to_clade.get(x, "Neoaves")), x))
dist_df = dist_df.loc[ordered, ordered]

ja_map = metadata.set_index("id")["common_name_ja"].to_dict()
labels_ja = [ja_map.get(x, x) for x in dist_df.index]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(dist_df, cmap="viridis",
            xticklabels=labels_ja, yticklabels=labels_ja,
            cbar_kws={"label": "p距離"}, ax=ax, square=True)
ax.set_title("31種間の遺伝的距離(系統順に並べ替え)")
plt.tight_layout()
plt.show()

### 5.4 近隣結合法（NJ法）で系統樹を作る

**Neighbor Joining（NJ）法** は距離行列から系統樹を作る代表的な方法です。1987年に斎藤と根井により発表され、現在も広く使われています。

In [ ]:
constructor = DistanceTreeConstructor()
nj_tree = constructor.nj(dm)

# 外群(ワニ)で根を打ち直す → 系統樹が「鳥類 vs ワニ」の構造になる
nj_tree.root_with_outgroup("Alligator_mississippiensis")

# 内部ノードのデフォルト名(Inner1, Inner2, ...)はラベル表示時の邪魔になるので消す
for clade in nj_tree.get_nonterminals():
    clade.name = None

print(f"NJ 系統樹を作成しました")
print(f"  末端ノード(種)の数: {nj_tree.count_terminals()}")

### 5.5 系統樹の描画（まずはシンプルに）

まずはシルエットなしで、和名ラベルを系統別の色で塗り分けたシンプルな樹を描いてみます。

系統樹の描画には **ETE3** を使います。ETE3 は系統樹の操作・可視化に特化したライブラリで、和名ラベルや系統色、各種のシルエット画像、ブートストラップ値などを柔軟にレイアウトできます。

In [ ]:
# === 系統別の色の設定 ===
clade_to_color = {
    "Palaeognathae": "#d95f02",   # オレンジ
    "Galloanserae": "#1b9e77",    # 緑
    "Neoaves":      "#7570b3",    # 紫
    "Outgroup":     "#444444",    # グレー
}
id_to_color = {row["id"]: clade_to_color.get(row["clade"], "#000000")
               for _, row in metadata.iterrows()}

# === ETE3 による系統樹描画ツールキット(以降のすべての樹で使い回す) ===

def bio_to_ete(bio_tree):
    """Biopython の系統樹を ETE3 の Tree に変換する。

    枝長(branch_length → dist)とブートストラップ値(confidence → support)を引き継ぐ。
    """
    def add(bio_clade, ete_node):
        for child in bio_clade.clades:
            n = ete_node.add_child(name=child.name or "",
                                   dist=child.branch_length or 0.0)
            if child.confidence is not None:
                n.support = child.confidence
            add(child, n)
    root = EteTree()
    root.name = bio_tree.root.name or ""
    root.dist = 0.0          # ルートの枝は除外(左端に伸びる余分な線を描かない)
    add(bio_tree.root, root)
    return root


_silhouette_cache = {}

def colored_silhouette(species, color):
    """黒シルエット PNG を系統色に塗り替えて一時ファイルに保存し、(パス, 表示幅) を返す。

    ETE3 の ImgFace はファイルパスを要求するため、色付き画像を一旦書き出して渡す。
    アスペクト比を保つよう、高さ36pxに対する幅を計算する。
    """
    key = (species, color)
    if key in _silhouette_cache:
        return _silhouette_cache[key]
    src = PHYLOPIC_DIR / f"{species}.png"
    if not src.exists():
        _silhouette_cache[key] = None
        return None
    img = mpimg.imread(src).astype(float)
    if img.max() > 1:
        img /= 255.0
    if img.ndim != 3 or img.shape[2] != 4:   # アルファ付き RGBA のみ対象
        _silhouette_cache[key] = None
        return None
    rgb = to_rgb(color)
    out = np.zeros_like(img)
    out[..., 0], out[..., 1], out[..., 2] = rgb[0], rgb[1], rgb[2]
    out[..., 3] = img[..., 3]              # アルファ(シルエットの形)は保持
    tmp_dir = Path(tempfile.gettempdir()) / "ete_silhouettes"
    tmp_dir.mkdir(exist_ok=True)
    dst = tmp_dir / f"{species}_{color.lstrip('#')}.png"
    mpimg.imsave(dst, out)
    h, w = img.shape[:2]
    result = (str(dst), max(1, int(36 * w / h)))
    _silhouette_cache[key] = result
    return result


def render_ete_to_png(tree, title, show_bootstrap=False, with_silhouettes=True,
                      width=1000, out_path=None):
    """Biopython 系統樹を ETE3 で描画して PNG に書き出し、ファイルパスを返す。

    - 葉:枝と和名ラベルを系統色で塗り、必要なら PhyloPic シルエットを並べる
    - 内部ノード:show_bootstrap=True ならブートストラップ値を支持率に応じた色で表示
    - 樹のタイトルと系統(clade)の凡例も ETE3 上で付与する
    """
    et = bio_to_ete(tree)

    def layout(node):
        if node.is_leaf():
            color = id_to_color.get(node.name, "#000000")
            ns = NodeStyle()
            ns["hz_line_color"] = color
            ns["hz_line_width"] = 2
            ns["size"] = 0
            node.set_style(ns)
            if with_silhouettes:
                sil = colored_silhouette(node.name, color)
                if sil:
                    node.add_face(ImgFace(sil[0], width=sil[1], height=36),
                                  column=0, position="aligned")
            label = ja_map.get(node.name, node.name)
            node.add_face(TextFace("  " + label, fsize=11, ftype=JP_FONT, fgcolor=color),
                          column=1, position="aligned")
        else:
            ns = NodeStyle()
            ns["hz_line_color"] = "#888888"
            ns["vt_line_color"] = "#888888"
            ns["size"] = 0
            node.set_style(ns)
            if show_bootstrap and node.support:
                bs = node.support
                bc = "#1b9e77" if bs >= 95 else ("#d95f02" if bs >= 70 else "#b22222")
                node.add_face(TextFace(f"{bs:.0f} ", fsize=8, fgcolor=bc),
                              column=0, position="branch-top")

    ts = TreeStyle()
    ts.show_leaf_name = False
    ts.layout_fn = layout
    ts.show_scale = True            # 枝長スケールバーを表示
    ts.title.add_face(TextFace(title, fsize=14, ftype=JP_FONT), column=0)

    # 系統(clade)の凡例
    for clade_name, c in clade_to_color.items():
        ts.legend.add_face(RectFace(16, 16, c, c), column=0)
        ts.legend.add_face(TextFace("  " + clade_name + "  ", fsize=10), column=1)
    if show_bootstrap:
        for lbl, c in [("BS ≥ 95", "#1b9e77"), ("BS 70–94", "#d95f02"), ("BS < 70", "#b22222")]:
            ts.legend.add_face(RectFace(16, 16, c, c), column=0)
            ts.legend.add_face(TextFace("  " + lbl + "  ", fsize=10), column=1)
    ts.legend_position = 1          # 左上

    if out_path is None:
        out_path = Path(tempfile.gettempdir()) / "ete_tree.png"
    et.render(str(out_path), tree_style=ts, w=width, dpi=120)
    return Path(out_path)


def draw_tree_ete(tree, title, show_bootstrap=False, with_silhouettes=True, width=500):
    """ETE3 で系統樹を描画し、ノートブックにインライン表示する。"""
    png = render_ete_to_png(tree, title, show_bootstrap=show_bootstrap,
                            with_silhouettes=with_silhouettes, width=width)
    display(IPyImage(filename=str(png)))


# === セクション 9:まずはシルエットなしのシンプルな樹 ===
draw_tree_ete(nj_tree,
              "鳥類 cytb の NJ 系統樹(外群=ミシシッピワニで Rooting)",
              with_silhouettes=False)

### 5.6 シルエット付きの系統樹

PhyloPic から得た各種のシルエット画像を末端に並べ、より直感的に読める樹を作ります。
シルエットは系統色で塗り分けます。

> PhyloPic ([phylopic.org](https://www.phylopic.org/)) は生物のシルエットを Creative Commons ライセンスで配布しているプロジェクトです。
> 帰属情報は `birds/phylopic/phylopic_attribution.tsv` を参照。

In [ ]:
# セクション 9 で定義した ETE3 ツールキットを使い、シルエット付きで描画する。
# 後のセクション(ML 系統樹)でも同じ名前で呼び出せるよう、薄いラッパーを用意しておく。
def draw_tree_with_silhouettes(tree, title, xlabel=None, show_bootstrap=False):
    """系統樹を PhyloPic シルエット付きで描画する(ETE3)。

    xlabel は後方互換のため受け取るが、ETE3 では枝長スケールバーが自動表示される。
    """
    draw_tree_ete(tree, title, show_bootstrap=show_bootstrap, with_silhouettes=True)


draw_tree_with_silhouettes(nj_tree, "鳥類 cytb の NJ 系統樹(PhyloPic シルエット付き)")

## 6. 本格編 — MAFFT でアラインメントし、IQ-TREE で最尤系統樹を推定

ここからは、研究現場で実際に使われている **業界標準ツール** で同じデータを解析し直します。

| ツール | 役割 | 入門編との違い |
| --- | --- | --- |
| **MAFFT** | マルチプルアラインメント | 配列を端で切るだけでなく、必要に応じて **ギャップ(挿入・欠失)** を入れて精密に揃える |
| **IQ-TREE** | 最尤法による系統樹推定 | 距離法ではなく、塩基置換モデルに基づく **確率的推論**。**ブートストラップ** で各枝の信頼度も算出 |

これらは Python ライブラリではなく **独立した実行ファイル(コマンドラインツール)** です。

### 6.1 MAFFTとIQ-TREE のインストール

環境に応じて以下のいずれかを実行してください(初回のみ)。

```bash
# Google Colab・Ubuntu/Debian
!apt-get install -y mafft iqtree

# conda 環境(より新しい IQ-TREE が手に入る)
conda install -c bioconda mafft iqtree

# macOS (Homebrew)
brew install mafft iqtree
```

> Note: 環境によって IQ-TREE のコマンド名は `iqtree`, `iqtree2`, `iqtree3` のいずれかです。
> 以下のコードでは自動検出し、バージョンに応じてオプションも切り替えます。

In [ ]:
# Colab で直接インストールしたいときは下のコメントを外す
# !apt-get install -y mafft iqtree

def find_command(candidates):
    for cmd in candidates:
        path = shutil.which(cmd)
        if path:
            return cmd, path
    return None, None

mafft_cmd, mafft_path = find_command(["mafft"])
iqtree_cmd, iqtree_path = find_command(["iqtree3", "iqtree2", "iqtree"])

print(f"MAFFT  : {mafft_path or '見つかりません'}")
print(f"IQ-TREE: {iqtree_path or '見つかりません'} (コマンド名: {iqtree_cmd})")

if not (mafft_cmd and iqtree_cmd):
    print("\n⚠ どちらかが見つからない場合は、上のセルに従ってインストールしてください。")
    print("  以下の MAFFT / IQ-TREE のセクションはスキップされ、NJ 系統樹のみ得られます。")

### 6.2 MAFFTで配列をアラインメント

**MAFFT** は速くて精度の高いマルチプルアラインメントツールです。
`--auto` オプションを使うと、配列数とコンピューター性能に応じた最適なアルゴリズムを自動で選んでくれます。

MAFFTに渡すために、まずヘッダーを整理したFASTAファイルを書き出します。

In [ ]:
# 整理済み ID で FASTA を再生成(MAFFT の入力に使う)
stripped_fasta = WORK_DIR / "birds_cytb_stripped.fasta"
clean_records = list(SeqIO.parse(FASTA_PATH, "fasta"))
for rec in clean_records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""
SeqIO.write(clean_records, stripped_fasta, "fasta")
print(f"ID 整理済み FASTA: {stripped_fasta}")

aligned_path = WORK_DIR / "birds_cytb_aligned.fasta"

if mafft_cmd:
    print("\nMAFFT を実行中(数秒で終わります)...")
    with open(aligned_path, "w") as f:
        subprocess.run(
            [mafft_cmd, "--auto", "--quiet", str(stripped_fasta)],
            stdout=f, stderr=subprocess.PIPE, check=True, text=True,
        )
    print(f"完了: {aligned_path}")
else:
    print("MAFFT がインストールされていません。スキップします。")

### 6.3 アラインメント結果を確認

MAFFT は挿入・欠失(ギャップ `-`)を入れて配列をきれいに揃えます。

In [ ]:
if aligned_path.exists():
    mafft_alignment = AlignIO.read(aligned_path, "fasta")
    print(f"配列数              : {len(mafft_alignment)}")
    print(f"カラム数(MAFFT後)  : {mafft_alignment.get_alignment_length()}")
    print(f"カラム数(切り詰めのみ): {alignment.get_alignment_length()}")
    diff = mafft_alignment.get_alignment_length() - alignment.get_alignment_length()
    print(f"  → MAFFT は {diff:+d} カラム分のギャップを追加")
    
    print("\n最初の60カラム × 5配列(ギャップ '-' に注目):")
    for rec in mafft_alignment[:5]:
        print(f"  {rec.id:30s} {str(rec.seq)[:60]}")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")
    mafft_alignment = None

### 6.4 保存性プロファイル（どの位置がどれだけ変わったか）

各カラムで**保存性スコア**（＝最も多い塩基がどれだけ多数派か）を計算し、配列に沿って描きます。

- 保存率の **高い** 領域 = タンパク質の機能的に重要な部位(変えると致命的)
- 保存率の **低い** 領域 = 中立進化に近く、種ごとに多様な部位

系統解析にはこの両方が役立ちます。低保存領域 = 系統情報量が多い、高保存領域 = アラインメントの「アンカー」として機能。

余力がある場合は、MAFFTの出力結果を[Jalview](https://www.jalview.org/) や [AliView](https://ormbunkar.se/aliview/) で開いてみましょう。保存性プロファイルで見えた「変動域」が実際にどう違うのか目で確認できます。

In [ ]:
if mafft_alignment is not None:
    aln_array = np.array([list(str(rec.seq).upper()) for rec in mafft_alignment])
    n_seqs, n_cols = aln_array.shape
    
    conservation = np.zeros(n_cols)
    for j in range(n_cols):
        col = aln_array[:, j]
        col_no_gap = col[col != "-"]
        if len(col_no_gap) > 0:
            _, counts = np.unique(col_no_gap, return_counts=True)
            conservation[j] = counts.max() / n_seqs
    
    # 30カラムの移動平均で滑らかに
    window = 30
    smoothed = np.convolve(conservation, np.ones(window)/window, mode="valid")
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(range(len(smoothed)), smoothed, color="steelblue", linewidth=1.5)
    ax.axhline(0.9, color="red", linestyle="--", alpha=0.5, label="保存率 90%")
    ax.axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="保存率 50%")
    ax.set_xlabel(f"アラインメント上のカラム位置({window}カラム移動平均)")
    ax.set_ylabel("保存性スコア")
    ax.set_title(f"cytb 遺伝子の保存性プロファイル({n_seqs}種、{n_cols}カラム)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n保存率 ≥ 90% のカラム: {(conservation >= 0.9).sum()} / {n_cols} "
          f"({100*(conservation >= 0.9).mean():.1f}%)")
    print(f"保存率 < 50% のカラム: {(conservation < 0.5).sum()} / {n_cols} "
          f"({100*(conservation < 0.5).mean():.1f}%)")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")

### 6.5 最尤法と IQ-TREE

**最尤法 (Maximum Likelihood, ML)** は、データを生み出す確率モデルに基づいて、
「観察された配列を最もよく説明する系統樹」を統計的に探す方法です。

**距離法 (NJ, UPGMA) との違い**

| | 距離法 (NJ) | 最尤法 (IQ-TREE) |
| --- | --- | --- |
| 入力 | 距離行列 | アラインメント|
| 塩基置換モデル | なし、または最初に固定 | データにfitするモデルを選択（JC, K2P, GTR + Γ + I など）|
| 計算量 | 軽い(数秒) | 重い（31種で数十秒〜2分）|
| 信頼度 | 通常はなし | ブートストラップで各枝の支持率を算出 |

**IQ-TREE のオプション**

- `-s` : 入力アラインメント
- `-m MFP` : ModelFinder Plus(最適な塩基置換モデルを自動選択)
- `-B 1000` (v2) / `-bb 1000` (v1) : UltraFast Bootstrap を 1000 回
- `-T AUTO` (v2) / `-nt AUTO` (v1) : スレッド数の自動設定
- `-redo` : 既存の出力を上書き(再実行時に便利)

ブートストラップとは、**アラインメントのカラムをランダムに重複抽出し直して系統樹を作り直す** ことを 1000 回繰り返し、
各枝が何 % の試行で再現されたかを「**支持率**」として返す仕組みです。
支持率が高い枝ほど、データの偶然のゆらぎに対して頑健であることを意味します。

In [ ]:
ml_treefile = Path(str(aligned_path) + ".treefile")

if iqtree_cmd and aligned_path.exists():
    # IQ-TREE のバージョンを検出してオプションを切り替える
    help_out = subprocess.run([iqtree_cmd, "--help"], capture_output=True, text=True)
    help_text = help_out.stdout + help_out.stderr
    is_modern = ("-B NUM" in help_text) or ("--prefix" in help_text)
    
    bootstrap_args = ["-B", "1000"] if is_modern else ["-bb", "1000"]
    threads_args = ["-T", "AUTO"] if is_modern else ["-nt", "AUTO"]
    print(f"IQ-TREE バージョン: {'v2/v3 系' if is_modern else 'v1 系'}")
    
    print(f"IQ-TREE を実行中(30秒〜2分)...")
    proc = subprocess.run(
        [iqtree_cmd, "-s", str(aligned_path), "-m", "MFP",
         *bootstrap_args, *threads_args, "-redo"],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        print("IQ-TREE 完了")
        print("\n生成されたファイル:")
        for f in sorted(WORK_DIR.glob("birds_cytb_aligned.fasta.*")):
            print(f"  {f.name}")
    else:
        print(f"IQ-TREE がエラーで終了しました:\n{proc.stderr[-500:]}")
else:
    print("IQ-TREE またはアラインメントが利用できないため、スキップします。")

### 6.6 採用されたモデルを確認

ModelFinder は何十種類もの塩基置換モデルから AIC/BIC で最良のものを選びます。
`.iqtree` レポートファイルに採用モデルが書かれているので、それを表示してみます。

In [ ]:
report_file = Path(str(aligned_path) + ".iqtree")
if report_file.exists():
    text = report_file.read_text()
    # 採用モデル行を探して表示
    for keyword in ["Best-fit model", "Model of substitution"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
    # 対数尤度と AIC
    for keyword in ["Log-likelihood of the tree", "Unconstrained log-likelihood",
                    "Akaike information criterion (AIC) score",
                    "Bayesian information criterion (BIC) score"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
else:
    print("IQ-TREE レポートがないため、このセルはスキップします。")

### 6.7 ML 系統樹の読み込みと描画

IQ-TREE の出力 `.treefile`(Newick 形式)を Biopython で読み込み、外群でルート付けします。
Newick の内部ノードラベルには **ブートストラップ支持率 (0〜100)** が格納されています。

In [ ]:
if ml_treefile.exists():
    ml_tree = Phylo.read(ml_treefile, "newick")
    ml_tree.root_with_outgroup("Alligator_mississippiensis")
    
    # Newick の内部ノードラベル(=ブートストラップ値の文字列)を confidence に移す
    for clade in ml_tree.get_nonterminals():
        if clade.name:
            try:
                clade.confidence = float(clade.name)
            except ValueError:
                pass
        clade.name = None
    
    bs_values = [c.confidence for c in ml_tree.get_nonterminals() if c.confidence is not None]
    print(f"ML 系統樹を読み込みました")
    print(f"  末端: {ml_tree.count_terminals()}")
    if bs_values:
        print(f"  ブートストラップ支持率: 平均 {np.mean(bs_values):.1f}, "
              f"範囲 [{min(bs_values):.0f}, {max(bs_values):.0f}]")
        print(f"  ≥ 95(強い支持)の枝: {sum(b >= 95 for b in bs_values)} / {len(bs_values)}")
else:
    print("ML 系統樹ファイルがないため、このセルはスキップします。")
    ml_tree = None

### 6.8 ML 系統樹をブートストラップ支持率付きで描画

各内部ノードにブートストラップ値を表示します。

- 🟢 緑 (≥ 95): 強い支持(その枝はほぼ確実)
- 🟠 橙 (70–94): 中程度の支持
- 🔴 赤 (< 70): 弱い支持(その枝は信頼できない)

支持率の低い枝は、データだけからは「どっちが正しいか決められなかった」ことを意味します。

In [ ]:
if ml_tree is not None:
    draw_tree_with_silhouettes(
        ml_tree,
        title="鳥類 cytb の ML 系統樹(IQ-TREE + UltraFast Bootstrap × 1000)",
        xlabel="塩基置換数 / サイト",
        show_bootstrap=True,
    )
else:
    print("ML 系統樹がないため、このセルはスキップします。")

### 6.9 NJとMLを並べて比較

2つの系統樹を並べて見比べてみましょう。おおまかな形状は一致していると思いますが、細かく見ると結構違いがあります。なぜ、違いが生まれたのか考察してみましょう。

In [ ]:
if ml_tree is not None:
    tmp = Path(tempfile.gettempdir())
    png_nj = render_ete_to_png(
        nj_tree, "NJ(切り詰めアラインメント + p距離)",
        with_silhouettes=False, out_path=tmp / "cmp2_nj.png")
    png_ml = render_ete_to_png(
        ml_tree, "ML(MAFFT + IQ-TREE 最尤法 + UFBoot 1000)",
        show_bootstrap=True, with_silhouettes=False, out_path=tmp / "cmp2_ml.png")

    fig, axes = plt.subplots(1, 2, figsize=(20, 14))
    for ax, png in zip(axes, [png_nj, png_ml]):
        ax.imshow(mpimg.imread(png))
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("ML 系統樹がないため比較できません。")

## 7. 考察 — 系統樹は何を語っているか

上で得た2つの系統樹（NJとML）を見比べながら、以下のテーマについて考えてみましょう。

### Q1. 3つの大きなグループに分かれた?

外群(ミシシッピワニ)から先にたどると、現代の鳥類分類の主要な見解と一致する **3 つの大グループ** に分かれているはずです。

| 系統 | 例 | 特徴 |
| --- | --- | --- |
| **Palaeognathae**(古顎類) | ダチョウ、キーウィ、レア、シギダチョウ | 飛べない走鳥が多い・原始的 |
| **Galloanserae**(キジカモ類) | ニワトリ、シチメンチョウ、ウズラ、カモ、ガン | 家禽として人類に重要 |
| **Neoaves**(新鳥類) | ハト、ツバメ、フクロウ、ペンギン、ワシ など | 現生鳥類の約 **95%** がここに含まれる |

ML樹で、この3分割を支える枝のブートストラップ支持率はどうなっていますか?

### Q2. 形質情報と進化の関係

- **飛べない鳥はただの鳥さ**
    - ペンギンとダチョウはどちらも飛べない鳥ですが、系統的に近いですか？
- **蜜吸う鳥も好きずき**
    - **ハチドリ**と**タイヨウチョウ**の好物はどちらも密で形もよく似ています。こちらの系統関係はどうでしょう？
- **君は海鳥アホウドリ**
    - **ペンギン** と **アホウドリ系の海鳥**（ミナミオオフルマカモメなど）は、どちらも海鳥という共通点はありますが、姿形はだいぶ異なるようにも見えます。これらは近縁でしょうか。それとも収斂でしょうか。
- **鳶は鷹を産むか**
    - ワシ(Haliaeetus)、ハヤブサ(Falco)、フクロウ(Bubo) の系統関係を見てみましょう。実は「ハヤブサ目はワシ目より、オウム目・スズメ目に近い」ことが近年の研究で分かっています。今回の樹はそれを再現できているでしょうか？